# Evaluating the multimodal RAG system

Fix retrieval before you touch prompts. If the right chunk is not in the top-k, no amount of prompt
engineering produces a correct answer — it just produces a confident wrong one.

Three measurements, in this order:
1. **recall@k** — is the correct chunk retrieved at all?
2. **faithfulness** — is every claim in the answer supported by the retrieved context?
3. **citation accuracy** — do the cited sources actually contain the claim?

In [ ]:
import os, json, sys, statistics
sys.path.insert(0, "../src")

os.environ["INDEX_NAME"] = "multimodal-rag"
os.environ["COLLECTION_ENDPOINT"] = "REPLACE-WITH-CollectionEndpoint-OUTPUT"
os.environ["BEDROCK_REGION"] = "us-east-1"

from shared.aoss import OpenSearchServerless
from shared import bedrock
from query.handler import retrieve, reciprocal_rank_fusion

client = OpenSearchServerless(os.environ["COLLECTION_ENDPOINT"])
print("indexed chunks:", client.count(os.environ["INDEX_NAME"]))

## 1. Build a question set from your own documents

Twenty hand-written questions beat two thousand synthetic ones. For each, record the document and
page that actually answers it — that is the ground truth recall is measured against.

In [ ]:
QUESTIONS = [
    {"question": "What is the stated RTO for the payments service?",
     "expected_document": "runbook.pdf", "expected_page": 4},
    {"question": "What does the architecture diagram show about cross-region failover?",
     "expected_document": "architecture.png", "expected_page": None},
    # ... add 15-20 more
]
len(QUESTIONS)

## 2. recall@k

Sweep k. The curve usually flattens around k=8-10; past that you are paying tokens for noise.

In [ ]:
def recall_at_k(questions, k):
    hits = 0
    for q in questions:
        results = retrieve(q["question"], k, {})
        found = any(
            r.get("documentId") == q["expected_document"]
            and (q["expected_page"] is None or r.get("page") == q["expected_page"])
            for r in results
        )
        hits += int(found)
    return hits / len(questions)

for k in (3, 5, 8, 12, 20):
    print(f"recall@{k}: {recall_at_k(QUESTIONS, k):.2f}")

### Which retriever is carrying the result?

Run each leg separately. If BM25 alone matches the fused score, the embeddings are not earning
their cost; if the multimodal leg never contributes, the image captions are probably too thin.

In [ ]:
def recall_by_leg(questions, k=8):
    legs = {"text_knn": 0, "multimodal_knn": 0, "bm25": 0, "fused": 0}
    index = os.environ["INDEX_NAME"]

    for q in questions:
        tv = bedrock.embed_text(q["question"])
        mv = bedrock.embed_multimodal(text=q["question"])

        results = {
            "text_knn": client.knn_search(index, "textVector", tv, k=k),
            "multimodal_knn": client.knn_search(index, "multimodalVector", mv, k=k),
            "bm25": client.keyword_search(index, q["question"], k=k),
        }
        results["fused"] = [{"_source": r} for r in reciprocal_rank_fusion(list(results.values()))[:k]]

        for leg, hits in results.items():
            docs = [h.get("_source", h).get("documentId") for h in hits]
            legs[leg] += int(q["expected_document"] in docs)

    return {leg: round(count / len(questions), 3) for leg, count in legs.items()}

recall_by_leg(QUESTIONS)

## 3. Faithfulness — an LLM judge over the retrieved context

Ask a model to check each claim against the context only. Anything unsupported is a hallucination,
even when it happens to be true in the world.

In [ ]:
JUDGE_SYSTEM = """You check whether an answer is supported by the given context.
Return strict JSON: {"claims": [{"claim": "...", "supported": true|false, "why": "..."}],
"faithfulness": 0.0-1.0}. A claim is supported only if the context states it."""

def judge(question, answer, context):
    result = bedrock.generate(
        system=JUDGE_SYSTEM,
        messages=[{"role": "user", "content": [{"type": "text", "text":
            f"Question: {question}\n\nContext:\n{context}\n\nAnswer:\n{answer}"}]}],
        max_tokens=1200, temperature=0.0,
    )
    return json.loads(result["text"][result["text"].find("{"): result["text"].rfind("}") + 1])

In [ ]:
scores = []
for q in QUESTIONS:
    hits = retrieve(q["question"], 6, {})
    context = "\n\n".join(f"[{i+1}] {h.get('text','')}" for i, h in enumerate(hits))
    # call your deployed API, or import generate_answer directly
    from query.handler import generate_answer
    answer = generate_answer(q["question"], hits)["answer"]
    verdict = judge(q["question"], answer, context)
    scores.append(verdict["faithfulness"])
    print(f"{verdict['faithfulness']:.2f}  {q['question'][:60]}")

print("mean faithfulness:", round(statistics.mean(scores), 3))

## 4. What to change when the numbers are bad

| Symptom | Usual cause | Fix |
|---|---|---|
| Low recall@20 | Chunks too large, or the content never got indexed | Check ingestion status in DynamoDB; drop chunk size to ~600 |
| Good recall@20, bad recall@5 | Ranking, not retrieval | Add a reranker, or raise the BM25 weight |
| Images never retrieved | Captions too short or generic | Lengthen the caption prompt; include surrounding page text |
| High recall, low faithfulness | Prompt or context ordering | Put the strongest source first; tighten the system prompt |
| Right answer, wrong citation | Position mapping | Check `_cited_positions` against the order sources were added |